# PPO 모델 학습

## PPO (Proximal Policy Optimization) 설정

PPO 알고리즘을 사용한 Actor-Critic 모델을 학습합니다.
- **특징**: 안정적인 정책 학습, 액션 마스킹 지원
- **아키텍처**: CNN 기반 Actor-Critic (공유 백본)
- **장점**: DQN 대비 더 안정적이고 샘플 효율적

In [ ]:
# 필요한 패키지 설치
!pip install torch torchvision gym matplotlib seaborn tensorboard
!pip install onnx onnxruntime

import torch
print(f"PyTorch 버전: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")

In [ ]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

import sys
import os

# 프로젝트 경로 설정
PROJECT_ROOT = '/content/drive/MyDrive/2048-rl-project'
TRAINING_PATH = os.path.join(PROJECT_ROOT, 'training')

# Python 경로에 추가
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
if TRAINING_PATH not in sys.path:
    sys.path.insert(0, TRAINING_PATH)

# 작업 디렉토리 변경
os.chdir(TRAINING_PATH)

print(f"  경로 설정 완료!")
print(f"  프로젝트 루트: {PROJECT_ROOT}")
print(f"  학습 경로: {TRAINING_PATH}")
print(f"  현재 작업 디렉토리: {os.getcwd()}")

In [ ]:
# PPO 모듈 import
print("PPO 모델 로딩 시작...")

from environment.game_2048 import Game2048Env
from models.PPO.ppo_agent import PPOAgent
from models.PPO.networks import count_parameters

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
from IPython.display import clear_output
import pandas as pd

print("PPO 모듈 로드 완료")

In [ ]:
# PPO 학습 설정
PPO_CONFIG = {
    # 네트워크 설정
    'network_type': 'cnn',  # 'cnn' 또는 'simple_cnn'
    
    # 학습 하이퍼파라미터
    'lr': 3e-4,
    'gamma': 0.99,
    'gae_lambda': 0.95,
    'clip_range': 0.2,
    'value_coef': 0.5,
    'entropy_coef': 0.01,
    'max_grad_norm': 0.5,
    
    # 버퍼 및 배치 설정
    'buffer_size': 2048,    # 롤아웃 버퍼 크기
    'batch_size': 64,       # 미니배치 크기
    'n_epochs': 10,         # 업데이트 에포크 수
    'target_kl': 0.015,     # KL divergence 조기 종료 임계값
    
    # 디바이스
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

print("🎯 PPO 학습 설정:")
for key, value in PPO_CONFIG.items():
    print(f"  {key}: {value}")

# 환경 생성
env = Game2048Env()

# PPO 에이전트 생성
print("\n🤖 PPO 에이전트 생성...")
ppo_agent = PPOAgent(**PPO_CONFIG)

print(f"📊 모델 파라미터: {count_parameters(ppo_agent.actor_critic):,}")
print(f"💻 디바이스: {ppo_agent.device}")
print(f"🧠 네트워크 타입: {PPO_CONFIG['network_type']}")

In [ ]:
# PPO 학습 함수
def train_ppo_agent(agent, env, episodes, rollout_steps=2048):
    """
    PPO 에이전트 학습 함수
    
    Args:
        agent: PPO 에이전트
        env: 게임 환경
        episodes: 학습할 에피소드 수
        rollout_steps: 롤아웃 스텝 수 (업데이트 주기)
    """
    print(f"🚀 PPO 학습 시작! (목표: {episodes} 에피소드)")
    start_time = time.time()
    best_score = 0
    best_highest = 0
    
    episode = 0
    total_steps = 0
    
    # 새로운 모니터 초기화
    ppo_monitor = TrainingMonitor()
    
    while episode < episodes:
        # 에피소드 시작
        state = env.reset()
        episode_reward = 0
        episode_score = 0
        episode_steps = 0
        
        while episode_steps < 5000:  # 최대 스텝 제한
            # 액션 선택
            valid_actions = env.get_valid_actions()
            if not valid_actions:
                break
                
            action, log_prob, value = agent.select_action(state, valid_actions, training=True)
            
            # 환경에서 액션 실행
            next_state, reward, done, info = env.step(action)
            
            # 경험 저장
            agent.store_experience(
                state=state,
                action=action,
                reward=reward,
                value=value,
                log_prob=log_prob,
                done=done,
                valid_actions=valid_actions
            )
            
            state = next_state
            episode_reward += reward
            episode_score = info['score']
            episode_steps += 1
            total_steps += 1
            
            if done:
                break
        
        # 에피소드 종료 처리
        # 마지막 상태의 가치 계산
        if not done:
            _, _, last_value = agent.select_action(state, env.get_valid_actions(), training=False)
        else:
            last_value = 0.0
            
        agent.episode_scores.append(episode_score)
        agent.finish_episode(last_value)
        
        # 모니터링 정보 업데이트
        ppo_monitor.add_episode(
            reward=episode_reward,
            score=episode_score,
            steps=episode_steps,
            loss=None,  # PPO는 에피소드별 loss가 아님
            highest_tile=info['highest']
        )
        
        episode += 1
        
        # 최고 기록 갱신
        if episode_score > best_score:
            best_score = episode_score
            agent.save_model(f'/content/drive/MyDrive/2048_models/ppo_model_best_score.pth')
            
        if info['highest'] > best_highest:
            best_highest = info['highest']
            agent.save_model(f'/content/drive/MyDrive/2048_models/ppo_model_best_tile.pth')
        
        # 주기적 로깅
        if episode % 10 == 0:
            stats = agent.get_stats()
            elapsed = time.time() - start_time
            
            print(f"Episode {episode:4d} | Score: {episode_score:5.0f} | Highest: {info['highest']:4.0f} | "
                  f"Reward: {episode_reward:7.2f} | Steps: {episode_steps:3d} | "
                  f"Buffer: {agent.rollout_buffer.size():4d} | Time: {elapsed/60:.1f}min")
            
            if 'avg_total_loss' in stats:
                print(f"   Loss: {stats['avg_total_loss']:.4f} | "
                      f"Policy: {stats['avg_policy_loss']:.4f} | "
                      f"Value: {stats['avg_value_loss']:.4f} | "
                      f"KL: {stats['avg_kl_divergence']:.4f}")
        
        # 평가 및 시각화
        if episode % 100 == 0:
            eval_results = evaluate_ppo_agent(agent, env, num_episodes=5)
            ppo_monitor.add_eval(episode, eval_results['avg_score'])
            
            print(f"🎯 평가 결과 (Episode {episode}):")
            print(f"   평균 점수: {eval_results['avg_score']:.1f}")
            print(f"   최고 점수: {eval_results['max_score']:.0f}")
            print(f"   평균 최고타일: {eval_results['avg_highest']:.0f}")
            print(f"   최고 타일: {eval_results['max_highest']:.0f}")
        
        # 주기적 시각화
        if episode % 50 == 0 and episode > 0:
            clear_output(wait=True)
            ppo_monitor.plot_progress(f"PPO Training Progress - Episode {episode}")
            
            stats = ppo_monitor.get_stats()
            print(f"📈 현재 통계 (최근 100 에피소드):")
            for key, value in stats.items():
                if isinstance(value, float):
                    print(f"  {key}: {value:.2f}")
                else:
                    print(f"  {key}: {value}")
            
            print(f"  총 스텝: {total_steps:,}")
            print(f"  학습 업데이트: {agent.steps_done}")
        
        # 주기적 모델 저장
        if episode % 500 == 0:
            agent.save_model(f'/content/drive/MyDrive/2048_models/ppo_checkpoint_{episode}.pth')
    
    # 최종 모델 저장
    agent.save_model(f'/content/drive/MyDrive/2048_models/ppo_final_action_masked.pth')
    
    print(f"\n✅ PPO 학습 완료!")
    print(f"🏆 최고 점수: {best_score}")
    print(f"🎲 최고 타일: {best_highest}")
    print(f"📚 총 학습 업데이트: {agent.steps_done}")
    
    return ppo_monitor


# PPO용 평가 함수
def evaluate_ppo_agent(agent, env, num_episodes=5):
    """PPO 에이전트 평가 함수"""
    total_scores = []
    total_steps = []
    highest_tiles = []
    
    for _ in range(num_episodes):
        state = env.reset()
        steps = 0
        while steps < 1000:
            valid_actions = env.get_valid_actions()
            if not valid_actions:
                break
                
            action, _, _ = agent.select_action(state, valid_actions, training=False)
            next_state, _, done, info = env.step(action)
            state = next_state
            steps += 1
            
            if done:
                total_scores.append(info['score'])
                total_steps.append(steps)
                highest_tiles.append(info['highest'])
                break
    
    return {
        'avg_score': np.mean(total_scores) if total_scores else 0,
        'avg_steps': np.mean(total_steps) if total_steps else 0,
        'avg_highest': np.mean(highest_tiles) if highest_tiles else 0,
        'max_score': max(total_scores) if total_scores else 0,
        'max_highest': max(highest_tiles) if highest_tiles else 0
    }

print("🔧 PPO 학습 및 평가 함수 준비 완료!")

In [ ]:
# PPO 학습 모니터링 클래스
class TrainingMonitor:
    def __init__(self):
        self.reset()

    def reset(self):
        self.episode_rewards = []
        self.episode_scores = []
        self.episode_steps = []
        self.episode_losses = []
        self.highest_tiles = []
        self.eval_scores = []
        self.eval_episodes = []

    def add_episode(self, reward, score, steps, loss, highest_tile):
        self.episode_rewards.append(reward)
        self.episode_scores.append(score)
        self.episode_steps.append(steps)
        self.episode_losses.append(loss)
        self.highest_tiles.append(highest_tile)

    def add_eval(self, episode, avg_score):
        self.eval_episodes.append(episode)
        self.eval_scores.append(avg_score)

    def plot_progress(self, title="PPO Training Progress"):
        if len(self.episode_rewards) < 10:
            return

        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        fig.suptitle(title, fontsize=16)

        # 보상
        axes[0,0].plot(self.episode_rewards, alpha=0.3, color='blue')
        axes[0,0].plot(pd.Series(self.episode_rewards).rolling(50).mean(), color='red')
        axes[0,0].set_title('Episode Rewards')
        axes[0,0].set_ylabel('Reward')

        # 점수
        axes[0,1].plot(self.episode_scores, alpha=0.3, color='green')
        axes[0,1].plot(pd.Series(self.episode_scores).rolling(50).mean(), color='red')
        axes[0,1].set_title('Episode Scores')
        axes[0,1].set_ylabel('Score')

        # 최고 타일
        axes[0,2].plot(self.highest_tiles, alpha=0.3, color='purple')
        axes[0,2].plot(pd.Series(self.highest_tiles).rolling(50).mean(), color='red')
        axes[0,2].set_title('Highest Tiles')
        axes[0,2].set_ylabel('Tile Value')

        # 스텝 수
        axes[1,0].plot(self.episode_steps, alpha=0.3, color='orange')
        axes[1,0].plot(pd.Series(self.episode_steps).rolling(50).mean(), color='red')
        axes[1,0].set_title('Episode Steps')
        axes[1,0].set_ylabel('Steps')
        axes[1,0].set_xlabel('Episode')

        # 평가 점수
        if self.eval_scores:
            axes[1,1].plot(self.eval_episodes, self.eval_scores, 'o-', color='darkgreen')
        axes[1,1].set_title('Evaluation Scores')
        axes[1,1].set_ylabel('Avg Score')
        axes[1,1].set_xlabel('Episode')

        # 빈 서브플롯 (PPO 특화 메트릭을 위해 남겨둠)
        axes[1,2].text(0.5, 0.5, 'PPO Metrics\n(Policy Loss, Value Loss, etc.)', 
                      ha='center', va='center', transform=axes[1,2].transAxes)
        axes[1,2].set_title('PPO Training Metrics')

        plt.tight_layout()
        plt.show()

    def get_stats(self):
        if not self.episode_scores:
            return {}

        recent_scores = self.episode_scores[-100:]
        recent_tiles = self.highest_tiles[-100:]

        return {
            'episodes': len(self.episode_scores),
            'avg_score': np.mean(recent_scores),
            'max_score': max(self.episode_scores),
            'avg_highest_tile': np.mean(recent_tiles),
            'max_highest_tile': max(self.highest_tiles),
            'avg_steps': np.mean(self.episode_steps[-100:])
        }

print("📊 PPO 학습 모니터링 시스템 준비 완료!")

In [ ]:
# PPO 모델 학습 실행
print("🚀 PPO 학습 시작!")

# 학습 실행
ppo_monitor = train_ppo_agent(
    agent=ppo_agent,
    env=env,
    episodes=2000,  # DQN보다 적은 에피소드로도 좋은 성능 기대
    rollout_steps=2048
)

print("✅ PPO 학습 완료!")

In [ ]:
# PPO 모델 성능 평가
print("📊 PPO 모델 최종 성능 평가 시작...")

class PPOPerformanceEvaluator:
    """PPO 모델용 성능 평가자"""
    
    def __init__(self, agent, env):
        self.agent = agent
        self.env = env
        
    def evaluate_comprehensive(self, num_games=100):
        """포괄적인 성능 평가"""
        print(f"🎮 PPO 모델 평가 시작 ({num_games} 게임)")
        
        results = {
            'scores': [],
            'highest_tiles': [],
            'steps': [],
            'game_details': []
        }
        
        for game_idx in range(num_games):
            game_result = self._play_single_game()
            
            results['scores'].append(game_result['final_score'])
            results['highest_tiles'].append(game_result['highest_tile'])
            results['steps'].append(game_result['steps'])
            results['game_details'].append(game_result)
            
            if (game_idx + 1) % 25 == 0:
                print(f"  진행률: {game_idx + 1}/{num_games} 게임 완료")
        
        return results
    
    def _play_single_game(self):
        """단일 게임 플레이"""
        state = self.env.reset()
        steps = 0
        
        while steps < 2000:
            valid_actions = self.env.get_valid_actions()
            if not valid_actions:
                break
                
            action, _, _ = self.agent.select_action(state, valid_actions, training=False)
            next_state, reward, done, info = self.env.step(action)
            
            state = next_state
            steps += 1
            
            if done:
                break
        
        return {
            'final_score': info['score'],
            'highest_tile': info['highest'],
            'steps': steps,
            'final_board': self.env.get_board().copy()
        }
    
    def analyze_and_visualize(self, results):
        """핵심 분석 및 시각화"""
        scores = np.array(results['scores'])
        highest_tiles = np.array(results['highest_tiles'])
        
        # 타일별 달성률 분석
        from collections import Counter
        tile_achievements = Counter(highest_tiles)
        
        print("\n🏆 타일 달성률:")
        total_games = len(scores)
        for tile in sorted(tile_achievements.keys(), reverse=True):
            count = tile_achievements[tile]
            percentage = (count / total_games) * 100
            print(f"  {tile:4d} 타일: {count:3d}게임 ({percentage:5.1f}%)")
        
        # 통계 출력
        print(f"\n📊 성능 요약:")
        print(f"  평균 점수: {np.mean(scores):.1f} ± {np.std(scores):.1f}")
        print(f"  최고 점수: {np.max(scores):.0f}")
        print(f"  평균 최고 타일: {np.mean(highest_tiles):.1f}")
        print(f"  최고 타일: {np.max(highest_tiles):.0f}")
        print(f"  2048 달성률: {np.mean(highest_tiles >= 2048) * 100:.1f}%")
        print(f"  1024 달성률: {np.mean(highest_tiles >= 1024) * 100:.1f}%")
        
        # 시각화
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        
        # 점수 분포
        axes[0,0].hist(scores, bins=30, alpha=0.7, color='skyblue', edgecolor='black')
        axes[0,0].set_title('점수 분포')
        axes[0,0].set_xlabel('점수')
        axes[0,0].set_ylabel('빈도')
        axes[0,0].axvline(np.mean(scores), color='red', linestyle='--', label=f'평균: {np.mean(scores):.1f}')
        axes[0,0].legend()
        
        # 최고 타일 분포
        tile_counts = Counter(highest_tiles)
        tiles = sorted(tile_counts.keys())
        counts = [tile_counts[tile] for tile in tiles]
        
        axes[0,1].bar(range(len(tiles)), counts, color='lightgreen', edgecolor='black')
        axes[0,1].set_title('최고 타일 분포')
        axes[0,1].set_xlabel('최고 타일')
        axes[0,1].set_ylabel('게임 수')
        axes[0,1].set_xticks(range(len(tiles)))
        axes[0,1].set_xticklabels([str(tile) for tile in tiles], rotation=45)
        
        # 점수 vs 최고 타일
        axes[1,0].scatter(highest_tiles, scores, alpha=0.6, color='purple')
        axes[1,0].set_title('점수 vs 최고 타일')
        axes[1,0].set_xlabel('최고 타일')
        axes[1,0].set_ylabel('점수')
        
        # 게임 길이 분포
        steps = [result['steps'] for result in results['game_details']]
        axes[1,1].hist(steps, bins=30, alpha=0.7, color='orange', edgecolor='black')
        axes[1,1].set_title('게임 길이 분포')
        axes[1,1].set_xlabel('스텝 수')
        axes[1,1].set_ylabel('빈도')
        axes[1,1].axvline(np.mean(steps), color='red', linestyle='--', label=f'평균: {np.mean(steps):.1f}')
        axes[1,1].legend()
        
        plt.tight_layout()
        plt.show()

# PPO 평가자 생성 및 평가 실행
ppo_evaluator = PPOPerformanceEvaluator(ppo_agent, env)
ppo_results = ppo_evaluator.evaluate_comprehensive(num_games=100)

# 분석 및 시각화
ppo_evaluator.analyze_and_visualize(ppo_results)

print("\n✅ PPO 모델 평가 완료!")

In [ ]:
# PPO 모델 ONNX 변환
print("🔄 PPO 모델 ONNX 변환 시작...")
ppo_agent.export_to_onnx(
    filepath='/content/drive/MyDrive/2048_models/ppo_model.onnx',
    input_shape=(4, 4, 16)
)
print("✅ PPO 모델 ONNX 변환 완료!")

print("\n🎉 모든 PPO 학습 과정이 완료되었습니다!")
print("📁 저장된 파일들:")
print("  - ppo_final_action_masked.pth (최종 모델)")
print("  - ppo_model_best_score.pth (최고 점수 모델)")
print("  - ppo_model_best_tile.pth (최고 타일 모델)")
print("  - ppo_model.onnx (ONNX 형식)")